# Le temps de sommeil chez les étudiants
Notre travail de recherche s'intéresse à l'étude d'une base de donnée de 500 étudiants ayant répondus à une enquête portant sur la qualité de leur sommeil. Chaque ligne correspond donc à un étudiant intérogé et chaque colonnes représentent des variables qualitatives et quantitative telle que la qualité de sommeil, le chronotype de l'individu, le temps qu'il estime d'écran, le temps d'études/révision hebdomadaire, les appareils qu'il utilise : smartphone, tablette, ipad...). 

*Dans la description, il est toutefois expliqué que ces données ont été généré artificiellement à des fins de travaux d'entraînements à la régression. Notons d'ailleurs que l'auteur de cette base de donnée a émis volontairément des désordre réel tel que : des valeurs manquantes, de mauvaises notation en lettre capitale ou minuscule (F/H, homme/femme) afin desimuler les particularités réelles de donnée trouvées dans le contexte de réelles enquêtes*

# Objectif de l'étude :
L'objectif sera de mettre en place un modèle de régression linéaire qui tendra à expliquer la qualité de sommeil des étudiants par le biais de variables tel que le chronotype, le temps de sommeil, le genre...
Nous ferons un premier travail de nettoyage des données, puis nous chercherons à trier les variables moins significatives avant de chercher à construire notre premier modèle de régression linéaire d'intérêt.


# Première visualisation de notre base de donnée :
Dans cette première partie nous nous occuperons de :

- visualiser les premières lignes
- explorer les variable explicatives
- afficher les statistiques descriptives de bases *(moyenne, quartiles, etc...)*

In [73]:
import pandas as pd
df = pd.read_csv('time_sleep.csv')
df.head(5) # nous venons ici d'afficher les 5 premières lignes de notre dataframe.

,student_id,age,gender,year_of_study,major_category,residence_type,chronotype,avg_bedtime_hour,avg_sleep_hours_weekday,weekend_sleep_delta_hours,...,naps_per_week,sleep_quality_score,exercise_days_per_week,energy_level,concentration_difficulty,weekly_study_hours,class_attendance_percent,has_part_time_job,semester_type,cgpa
0,STU2001,21,Male,3,Humanities,With Family,Early Bird,22.3,7.6,2.0,...,0,10.0,7,5.0,7.0,15.6,80.0,No,Spring,2.99
1,STU2002,20,Female,2,STEM,Dormitory,Intermediate,23.1,5.9,2.7,...,3,7.0,3,6.0,7.0,17.5,78.9,No,Summer,2.88
2,STU2003,20,Male,4,Social Sciences,With Family,Intermediate,23.4,8.4,1.8,...,1,9.0,2,5.0,1.0,19.0,69.7,Yes,Spring,3.15
3,STU2004,22,F,1,Humanities,Off-campus Apartment,Night Owl,0.9,5.1,2.4,...,3,6.0,4,3.0,7.0,7.3,59.1,No,Fall,2.27
4,STU2005,23,Female,4,Social Sciences,Off-campus Apartment,Early Bird,23.4,5.9,2.0,...,2,7.0,6,8.0,3.0,23.6,79.9,No,Summer,2.74


# Les problème d'encodage
Le premier constat que l'on peut faire concerne la colonne genre. En effet, on constate que les réponses sur le genre contiennent plus de trois ou quatre choix dû principalement aux personnes interogés qui n'ont pas forcément utilisé les mêmes notations, cela nécessite donc de ré-encoder par exemple Female par F etc...

Après avoir corrigé ce problème, nous afficherons le jeu de donnée avec les valeurs corrigés

*Nous avons fait le choix de laisser en français le nom des genres exceptionnelement pour cette variable)*

In [74]:
df['gender'] = df['gender'].replace({
    'F': 'Femme',
    'female': 'Femme',
    'Female': 'Femme',
    'M': 'Homme',
    'male': 'Homme',
    'Male': 'Homme',
    'Non-binary': 'Non-binaire',
    'Prefer not to say': 'Préfère ne pas répondre'
})
df


,student_id,age,gender,year_of_study,major_category,residence_type,chronotype,avg_bedtime_hour,avg_sleep_hours_weekday,weekend_sleep_delta_hours,...,naps_per_week,sleep_quality_score,exercise_days_per_week,energy_level,concentration_difficulty,weekly_study_hours,class_attendance_percent,has_part_time_job,semester_type,cgpa
0,STU2001,21,Homme,3,Humanities,With Family,Early Bird,22.3,7.6,2.0,...,0,10.0,7,5.0,7.0,15.6,80.0,No,Spring,2.99
1,STU2002,20,Femme,2,STEM,Dormitory,Intermediate,23.1,5.9,2.7,...,3,7.0,3,6.0,7.0,17.5,78.9,No,Summer,2.88
2,STU2003,20,Homme,4,Social Sciences,With Family,Intermediate,23.4,8.4,1.8,...,1,9.0,2,5.0,1.0,19.0,69.7,Yes,Spring,3.15
3,STU2004,22,Femme,1,Humanities,Off-campus Apartment,Night Owl,0.9,5.1,2.4,...,3,6.0,4,3.0,7.0,7.3,59.1,No,Fall,2.27
4,STU2005,23,Femme,4,Social Sciences,Off-campus Apartment,Early Bird,23.4,5.9,2.0,...,2,7.0,6,8.0,3.0,23.6,79.9,No,Summer,2.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,STU2496,18,Homme,4,Humanities,Dormitory,Intermediate,22.5,7.5,0.4,...,0,8.0,4,7.0,4.0,10.1,90.8,No,Fall,3.21
496,STU2497,22,Homme,2,Health Sciences,Dormitory,Night Owl,0.8,6.2,2.4,...,2,10.0,4,6.0,5.0,17.2,70.7,No,Spring,2.87
497,STU2498,20,Femme,5,Health Sciences,With Family,Early Bird,22.0,7.1,1.1,...,3,10.0,1,8.0,2.0,21.7,79.8,No,Spring,3.54
498,STU2499,22,Homme,3,STEM,Dormitory,Early Bird,23.6,9.0,2.3,...,0,10.0,7,7.0,1.0,19.3,97.8,Yes,Spring,3.67


# Le problème des données manquantes
Dans l'analyse de donnée, le problème des données manquantes est fréquent et peut poser un frein à l'étude économétrique de variables et modèles. En effet, lorsque NULL apparaît, très souvent, python ne peut faire de calcul et un message d'erreur survient. Alors comment traîter ce problème ? 

 La première solution est celle que l'on évite en général d'appliquer car elle réduit de manière plus ou moins importante le nombre d'individus dans notre jeu de donnée. Il s'agit de la suppression des lignes contenant ces valeurs manquantes. Le problème étant que l'on réduit significativement l'importance de nos modèles car la population est moins représentative. Les valeurs mathématiques trouvées se retrovent aussi impactées. La deuxième solution est celle que nous allons priviligier pour la suite de notre travail : l'imputation. Cette méthode consiste à imputer une certaine valeur à ces valeurs manquantes selon un critère choisi préalablement. Cela nécessite tout d'abord de connaître le type de ces valeurs manquantes, c'est-à-dire s'il s'agit de données quantitatives ou bien qualitatives. Après cela, nous somme capable pour chaque type, de déterminer une imputation par une méthode adéquate. Notons que le choix de la méthode d'imputation compte tout autant, et certaines méthode ne se valent pas. 
 
 > Pour les variables quantitatives, remplacer les valeurs manquantes par la moyenne ou la médiane restent des méthodes supérieur à la méthode de suppression de ligne mais d'un point de vue économétrique, elle reste moins bonne que d'autres existantes. En effet, par le biais d'indicateur tel que le RMSE qui mesure la différence entre les valeurs prédites et les valeurs réelles élevées au carré, on a très souvent remarqué que de manière générale, les scores de RMSE étaient bien supérieur dans les modèles avancés tel que remplacer les variables par les forêts aléatoires, les KNN ou encore les régressions. En tout point, le RMSE était  bien plus élevé que ceux des modèles par la moyenne ou la médiane. Dans notre travail, nous choisirons donc la méthode des KNN. Ce choix reste avant tout personnel dans notre démarche de travail mais il aurait tout à fait été possible d'utiliser les KNN. 

Pour les variables qualitatives, remplacer les valeurs manquantes par le mode est le choix recommandé, ou du moins le plus facile à appliquer, tout comme la moyenne pour les variables qualitatives... mais elle pose un principal problème. En utilisant des indicateurs comme l'accuracy ou le F1-score, le taux de réussite du modèle à prédire correctement ou du mieux possible, ceux à quoi ces valeurs manquantes auraient pu se rapprocher reste très faible. En clair, ce principe d'ignorer les modalités minoritaires et imputer pour toutes, par des valeurs synthétiques copiant la modalité la plus représenté à pour conséquence d'augmenter le taux d'erreur et de biais que cette méthode engendre. C'est pour cela qu'en général les modèles plus avancé touchant au machine learning, la regression ou KNN reste un meilleur car, on tient davantage compte de ces classe minoritaire en s'appuyant sur la proximité de certaines données croisées avec d'autres variables (de nature qualitatives ou quantitatives). Un exercice intéressant que nous avons essayé a été de constater la différence de score d'accuracy sur la base de donnée seaborn "pinguin" qui s'intéresse à des populations de pingouin et on se rend compte qu'en enlevant de manière aléatoire des données sur la race/l'espèce de pingouin, le modèle faisait moins d'erreur de prédiction en utilisant les KNN ou la regression logistique plutôt que l'imputation  par le mode car justement, les valeurs sur la taille des ailes, du bec, des nageoires pouvaient directement aider le modèle à identifier l'espèce de départ. Cela montre bien que parfois, une modalité pour un individus peut être lié d'autres réponses que ce dernier à inscrit, et le modèle pourra plus facilement repérer ces réponses. 

> Dans le cadre de cette études, on remplacera donc les valeurs manquantes des variables qualitatives par des modalités issus de l'imputation par les KNN. En effet, elle a pour avantage d'être plus précise que le mode (comme expliqué précédemment) et moins longue qu'en appliquant un modèle de machine learning qui peut prendre être plus chonophage à faire tourner.

## Repérer les valeurs manquantes
Comme énnoncé précédemment, nous utiliserons les commandes python pour rechercher au sein du dataframe où sont localisé les valeurs manquantes. Puis, selon leur nature, on sera en capacité d'appliquer l'imputation choisi. Rappelons-le, nous choissons pour les variables qualitatives le remplacement par le mode et pour les variables quantitatives le remplacement par les KNN.

In [75]:
df.isna().sum().sum()
print(df.isna().sum())

student_id                    0
age                           0
gender                        0
year_of_study                 0
major_category                0
residence_type                0
chronotype                    0
avg_bedtime_hour              0
avg_sleep_hours_weekday       0
weekend_sleep_delta_hours     0
pre_sleep_screen_time_min     0
daily_screen_time_hours       0
primary_device                0
device_free_before_bed        9
caffeine_drinks_per_day       0
naps_per_week                 0
sleep_quality_score           6
exercise_days_per_week        0
energy_level                 15
concentration_difficulty     14
weekly_study_hours            0
class_attendance_percent      0
has_part_time_job             0
semester_type                 0
cgpa                          7
dtype: int64


Après étude de ces valeurs manquantes, voici donc les variables concernées : 

- device free before bed (variable qualitative nominale, binaire)
- sleep_quality_score (variable quantitative discrète)
- energy_level (variable quantitative discrète)
- concentration_difficulty (variable quantitative discrète)
- cgpa (variable quantitative continue)

On dénombre au total 51 valeurs manquantes.


## Imputation par les KNN (pour les variables qualitatives)
Pour la variable device free before bed, elle est la seule variables qualitative à contenir des données manquantes. Nous appliquerons donc nous l'avions dit sur le principe des K plus proche voisins.

In [76]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Si 'df' n'est pas déjà défini dans le noyau courant, on le charge et nettoie 'gender'
if "df" not in globals():
  df = pd.read_csv("time_sleep.csv")
  df["gender"] = df["gender"].replace({
      "F": "Femme",
      "female": "Femme",
      "Female": "Femme",
      "M": "Homme",
      "male": "Homme",
      "Male": "Homme",
      "Non-binary": "Non-binaire",
      "Prefer not to say": "Préfère ne pas répondre",
  })

cible = "device_free_before_bed"

# 1. Variables explicatives (numériques + qualitatives utiles, hors cible et identifiant)
cols_num = [c for c in df.select_dtypes(include="number").columns if c != "cgpa"]
cols_quali = [
    c
    for c in df.select_dtypes(exclude="number").columns
    if c not in ["student_id", cible]
]

# 2. Prétraitement : standardisation pour le numérique, encodage one-hot pour le qualitatif
preprocesseur = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            cols_num,
        ),
        (
            "quali",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cols_quali,
        ),
    ]
)

# 3. Entraînement du classifieur KNN uniquement sur les lignes où la cible est observée
masque_obs = df[cible].notna()
masque_manquant = df[cible].isna()

if masque_manquant.any():
  X_train = df.loc[masque_obs, cols_num + cols_quali]
  y_train = df.loc[masque_obs, cible]
  X_pred = df.loc[masque_manquant, cols_num + cols_quali]

  knn_clf = Pipeline([
      ("prep", preprocesseur),
      ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance")),
  ])

  knn_clf.fit(X_train, y_train)
  df.loc[masque_manquant, cible] = knn_clf.predict(X_pred)

print(df[cible].value_counts(dropna=False))

device_free_before_bed
No     376
Yes    124
Name: count, dtype: int64


## Imputation par les KNN (pour les variables quantitatives)

In [77]:
import pandas as pd
from sklearn.impute import KNNImputer
# PARTIE 1 : Calcul des voisins sur l'ensemble des variables numériques
cols_num = df.select_dtypes(include='number').columns
imputer = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(imputer.fit_transform(df[cols_num]), columns=cols_num, index=df.index)

# La variable df_knn est la représentation d'une copie de notre dataframe de départ mais à la différence 
# que les valeurs manquantes de toutes les colonnes numériques (int), ont été remplacé par le KNN des 5 plus proches voisins. 
# Ainsi, nous avons simplement à dire à python de prendre les valeurs de nos colonnes de ce dataframe avec imputation 
# et l'appliquer au dataframe d'origine :

# PARTIE 2 : Imputation colonne par colonne (avec le formatage adapté à chacune)
df['sleep_quality_score'] = df_knn['sleep_quality_score'].round()
df['energy_level'] = df_knn['energy_level'].round()
df['concentration_difficulty'] = df_knn['concentration_difficulty'].round()
df['cgpa'] = df_knn['cgpa'].round(2)
print(df.isna().sum().sum())
#Le résultat affiche bien 0, le dataframe ne contient désormais plus aucune valeurs manquante qui pourraient impacter nos analyses.

0


## Statistiques descripives :
Après avoir réglé le problème des valeurs manquantes qui posait un frein à notre travail, il sera désormais possible de commencer notre analyser. Rappelons bien que notre objectif est d'expliquer la variable CGPA qui mesure le sommeil des étudiants avec une valeur quantitatives similaire à un score. Il va être important de commencer dans un pemier temps de montrer les statistiques descriptives, car ces dernières nous donne une vue d'ensemble sur nos données et cela resté la première porte d'entrée dans un travail de recherche. 

### Statistiques sur le dataframe :

In [78]:
df.shape # le dataframe contient 500 lignes et 25 variables explicatives.
int(df.isna().sum().sum()) # aucune valeurs manquantes
df.dtypes.value_counts() # On dénombre 16 variables de nature quantitatives et 9 variables de nature qualitatives


float64    10
str         9
int64       6
Name: count, dtype: int64

Commentaire : Le jeu de donnée contient 500 lignes et 25 variables. Aucune valeurs manquantes n'a été trouvé après imputation, ce qui montre que notre résultat à méthode a bien fonctionné. Enfin On dénombre 16 variables de nature quantitatives et 9 variables de nature qualitatives.

### Statistique sur les variables :

In [79]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,500.0,20.2600,1.994079,17.00,19.000,20.00,22.000,27.00
year_of_study,500.0,2.9540,1.405644,1.00,2.000,3.00,4.000,5.00
avg_bedtime_hour,500.0,12.7782,10.772656,0.00,1.100,21.50,22.800,24.00
avg_sleep_hours_weekday,500.0,6.6082,1.435784,1.50,5.700,6.60,7.700,10.50
weekend_sleep_delta_hours,500.0,1.5406,0.965441,-1.00,0.900,1.50,2.200,4.40
pre_sleep_screen_time_min,500.0,38.3880,23.573478,0.00,22.000,35.50,54.000,240.00
daily_screen_time_hours,500.0,5.8270,2.046193,0.50,4.475,5.80,7.200,19.50
caffeine_drinks_per_day,500.0,1.5020,1.273279,0.00,1.000,1.00,2.000,7.00
naps_per_week,500.0,1.4580,1.331082,0.00,0.000,1.00,2.000,7.00
sleep_quality_score,500.0,7.2800,2.123822,1.00,6.000,7.00,9.000,10.00


Par le biais de la commande **df.describe().T**, on affiche en sortie le résultat en ligne des différentes variables de notre dataframe et en colonne les résultats pour chaque variables concernant : la médiane, la moyenne, les quartiles, la valeur minimum/maximum ainsi que l'écart-type. 

# Mise en place d'une analyse bi-variée

Après avoir présenté brièvement les statistiques descriptives de notre jeu de données, il ne nous reste plus qu'à définir clairement les objectifs de notre études et notamment la relation qu'on cherche à mettre en évidence. Rappelons-le, notre base de donnée s'interesse au sommeil des étudiant qui ce dernier nous en informe de son sens par le biais de la variable **cgpa**. L'intérêt pour nous est donc de faire un tri en amont de chaque variable avec notre variable d'intérêt car nous ne pouvons mettre en place une regression linéaire avec 25 variables, il est préférable de se demander si nous pouvons réduire ce chiffre par le biais d'indicateurs économétriques de référence. Le plus utilisé et pertinent sera le coefficient de Pearson pour les relations avec des variables quantitatives. En revanche, pour les variables qualitatatives on utilisera l'ANOVA.

In [80]:
from scipy import stats

cible = "sleep_quality_score"
vars_quanti = df.select_dtypes(include="number").columns.drop(cible)
vars_quali = df.select_dtypes(exclude="number").columns.drop("student_id")

# TEST DE PEARSON
print(f"=== CORRÉLATIONS DE PEARSON (vs {cible}) ===")
for col in vars_quanti:
  r, p_val = stats.pearsonr(df[col], df[cible])
  print(f"{col} : r = {round(r, 3)} | p-value = {round(p_val, 4)}")

# TEST ANOVA
print(f"\n=== TESTS ANOVA (vs {cible}) ===")
for col in vars_quali:
  groupes = [
      df[df[col] == categorie][cible] for categorie in df[col].unique()
  ]
  f_stat, p_val = stats.f_oneway(*groupes)
  print(f"{col} : F = {round(f_stat, 3)} | p-value = {round(p_val, 4)}")

=== CORRÉLATIONS DE PEARSON (vs sleep_quality_score) ===
age : r = 0.03 | p-value = 0.5087
year_of_study : r = -0.06 | p-value = 0.1795
avg_bedtime_hour : r = 0.297 | p-value = 0.0
avg_sleep_hours_weekday : r = 0.741 | p-value = 0.0
weekend_sleep_delta_hours : r = -0.348 | p-value = 0.0
pre_sleep_screen_time_min : r = -0.439 | p-value = 0.0
daily_screen_time_hours : r = -0.014 | p-value = 0.7484
caffeine_drinks_per_day : r = -0.258 | p-value = 0.0
naps_per_week : r = -0.291 | p-value = 0.0
exercise_days_per_week : r = -0.003 | p-value = 0.9393
energy_level : r = 0.316 | p-value = 0.0
concentration_difficulty : r = -0.237 | p-value = 0.0
weekly_study_hours : r = -0.045 | p-value = 0.3198
class_attendance_percent : r = 0.28 | p-value = 0.0
cgpa : r = 0.467 | p-value = 0.0

=== TESTS ANOVA (vs sleep_quality_score) ===
gender : F = 4.484 | p-value = 0.0041
major_category : F = 0.413 | p-value = 0.8708
residence_type : F = 1.337 | p-value = 0.2617
chronotype : F = 92.164 | p-value = 0.0
pri

D'après les résultats obtenus, en choisissant un seuil de significativité de $\alpha = 0.05$, sur les 23 variables explicatives testées (en excluant l'identifiant student_id et la variable cible sleep_quality_score), on constate qu'une grande partie ne peut pas être retenue. En effet, pour des variables telles que :

- l'âge (age)
- l'année d'études (year_of_study)
- le temps d'écran quotidien (daily_screen_time_hours)
- le nombre de jours d'exercice physique (exercise_days_per_week)
- les heures d'étude hebdomadaires (weekly_study_hours)
ou encore des facteurs qualitatifs comme le type de résidence, l'appareil principal utilisé (primary_device) et le fait d'avoir un emploi à temps partiel (has_part_time_job), les tests statistiques réalisés (ANOVA et corrélations de Pearson) affichent des $p$-values bien supérieures à $0.05$, ce qui ne permet pas de rejeter l'hypothèse nulle d'indépendance avec notre variable cible, le sleep_quality_score.

Par conséquent, ces 11 facteurs ne montrent pas d'association linéaire ou de différence de moyenne statistiquement significative dans cet échantillon, et sont donc écartés de la suite de la modélisation au profit des 12 variables validées : 10 variables quantitatives (avg_bedtime_hour, avg_sleep_hours_weekday, weekend_sleep_delta_hours, pre_sleep_screen_time_min, caffeine_drinks_per_day, naps_per_week, energy_level, concentration_difficulty, class_attendance_percent et cgpa) ainsi que 2 variables qualitatives (gender et chronotype).

# Analyse de multicolinéarité
Dans cette partie, l'objectif est ici de chercher à savoir si un lien peut exister entre certaines de nos variables que nous avons retenues. Pour cela, on utilisera un indicateur économétrique qui s'appelle le VIF. Ce dernier a pour but de calculer à quel point la variance d'un coefficient de régression est augmentée en raison de la colinéarité avec les autres variables explicatives. 

Pour expliquer ce concept avec nos mots sans être trop penché sur les mathématiques derrières, nous en avons compris que même si la régression de départ qui cherchait à expliquer cpga est notre objectif final, le but est de savoir tout d'abord si une variable est lié aux 11 autres. Pour cela, on pourrait faire une sorte de sous-regression où on explique cete variable pa les 11 autres, et faire 12 sous-régression au total pour toutes... Cela serait trop long. Donc on a compris qu'avec l'outil de la matrice de corrélation, cette dernière permet d'afficher avec un coefficient entre 0 et 1, un couple de variable précis. Mais ce qui a été plus complexe c'est l'explication d'un certain théorème s'appelant le théorème de Shur qui explique qu'en algèbre linéaire, inverser une matrice permet de résoudre et trouver les solutions pour les différents systèmes. Comme chaque couple de variables sont inter-connectés, l'inversion de la matrice de corrélation permet de résoudre les 12 régressions que nous voulions au départ calculer. Le VIF apparaît sur les nombres situés sur la diagonale de cette matrice inversée.

> Le VIF évalue donc le degré de redondance d'une variable par rapport à l'ensemble des autres régresseurs. Un score VIF élevé (généralement supérieur à 5) signale une multicolinéarité problématique susceptible de fausser ou d'instabiliser nos estimations, tandis que des valeurs faibles confirment l'indépendance relative de nos variables et valident leur introduction conjointe dans le modèle de régression multiple.

In [81]:
import numpy as np
import pandas as pd

vars_quanti_retenues = [
    "avg_bedtime_hour",
    "avg_sleep_hours_weekday",
    "weekend_sleep_delta_hours",
    "pre_sleep_screen_time_min",
    "caffeine_drinks_per_day",
    "naps_per_week",
    "energy_level",
    "concentration_difficulty",
    "class_attendance_percent",
    "cgpa",
]

vars_quali_retenues = [
    "gender",
    "chronotype",

X = pd.get_dummies(
    df[vars_quanti_retenues + vars_quali_retenues],
    columns=vars_quali_retenues,
    drop_first=True,
    dtype=float,
)

vif_data = pd.DataFrame({
    "Variable": X.columns,
    "VIF": np.linalg.inv(X.corr().values).diagonal(),
})

print(vif_data.sort_values(by="VIF", ascending=False).to_string(index=False))

SyntaxError: '[' was never closed (2566027193.py, line 17)

D'après certaines études, on admet un seuil de tolérance pour les valeurs du VIF comprises entre 1 et 5. Comme nous le remarquons, aucune de nos variables explicaives est sujet à un problème de multicolinéarité. nous n'aurons donc aucune autres variables à enlever.

# Premier modèle de régression

In [ ]:
import statsmodels.api as sm

X = df[["avg_bedtime_hour",
        "avg_sleep_hours_weekday",
        "weekend_sleep_delta_hours",
        "pre_sleep_screen_time_min",
        "caffeine_drinks_per_day",
        "naps_per_week",
        "energy_level",
        "concentration_difficulty",
        "weekly_study_hours",
        "class_attendance_percent",
        "cgpa"]]
y = df["sleep_quality_score"]
X_sm = sm.add_constant(X)
modele_sm = sm.OLS(y, X_sm).fit()
print(modele_sm.summary())

                             OLS Regression Results                            
Dep. Variable:     sleep_quality_score   R-squared:                       0.643
Model:                             OLS   Adj. R-squared:                  0.635
Method:                  Least Squares   F-statistic:                     79.89
Date:                 Tue, 24 Nov 2026   Prob (F-statistic):          1.11e-101
Time:                         02:03:14   Log-Likelihood:                -828.09
No. Observations:                  500   AIC:                             1680.
Df Residuals:                      488   BIC:                             1731.
Df Model:                           11                                         
Covariance Type:             nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const       

# Analyse des résultats : 
## Données globales de la première régression :
Tout d'abord, en analysant ces résultats, on obtient un score de 0.643 de R² ajusté, ainsi, le modèle explique environ 64,3 % de la variance du score de qualité de sommeil, ce fort pourcentage rend compte d'un niveau intéressant de robustesse du modèle. 
Ensuite, la F-statistique, qui mesure la significativité globale de la régression est grandement significatif avec une valeur de 1.11e-101.

Elle teste l'hypothèse nulle $H_0$ selon laquelle tous les coefficients des variables explicatives sont simultanément nuls (sauf la constante) :

- $H_0 : \beta_1 = \beta_2 = \dots = \beta_{11} = 0$

avec cette valeur extrèmement faible, on rejette donc H0 et on reconnait donc que l'ensemble de nos variables explicatives a bien un effet conjoint statistiquement significatif sur sleep_quality_score. Les variables retenues apportent donc bien une réelle information sur la qualité du sommeil. On accepte donc bien l'hypothèse H1 qui est la suivante : 

- $H_1 : \exists\, j \in \{1, 2, \dots, 11\} \quad \text{tel que} \quad \beta_j \neq 0$


In [ ]:
#| echo: false
import pandas as pd
import statsmodels.api as sm

vars_quant = [
    "avg_bedtime_hour",
    "avg_sleep_hours_weekday",
    "weekend_sleep_delta_hours",
    "pre_sleep_screen_time_min",
    "caffeine_drinks_per_day",
    "naps_per_week",
    "energy_level",
    "concentration_difficulty",
    "weekly_study_hours",
    "class_attendance_percent",
    "cgpa",
]

X = sm.add_constant(df[vars_quant], has_constant="add")
y = df["sleep_quality_score"]

results = sm.OLS(y, X).fit()

ic_95 = results.conf_int(alpha=0.05)

df_coef = pd.DataFrame({
    "Coefficient": results.params,
    "Erreur type": results.bse,
    "Statistique t": results.tvalues,
    "p-value": results.pvalues,
    "IC 95% inf": ic_95[0],
    "IC 95% sup": ic_95[1],
})

df_coef["Significatif (5%)"] = df_coef["p-value"].apply(
    lambda p: "Oui ***" if p < 0.001 else ("Oui *" if p < 0.05 else "Non")
)

# 4. Mise en forme lisible
df_affichage = df_coef.copy()
cols_num = ["Coefficient", "Erreur type", "Statistique t", "IC 95% inf", "IC 95% sup"]
df_affichage[cols_num] = df_affichage[cols_num].round(4)
df_affichage["p-value"] = df_affichage["p-value"].apply(
    lambda p: "< 0.0001" if p < 0.0001 else f"{p:.4f}"
)

df_affichage

,Coefficient,Erreur type,Statistique t,p-value,IC 95% inf,IC 95% sup,Significatif (5%)
const,-0.2405,0.7154,-0.3361,0.7369,-1.6462,1.1653,Non
avg_bedtime_hour,-0.0036,0.0060,-0.6008,0.5482,-0.0153,0.0082,Non
avg_sleep_hours_weekday,0.8193,0.0592,13.8445,< 0.0001,0.7030,0.9356,Oui ***
weekend_sleep_delta_hours,-0.1048,0.0665,-1.5758,0.1157,-0.2355,0.0259,Non
pre_sleep_screen_time_min,-0.0207,0.0027,-7.6977,< 0.0001,-0.0260,-0.0154,Oui ***
caffeine_drinks_per_day,-0.1861,0.0468,-3.9779,< 0.0001,-0.2780,-0.0942,Oui ***
naps_per_week,-0.0473,0.0470,-1.0052,0.3153,-0.1397,0.0451,Non
energy_level,0.0072,0.0325,0.2220,0.8244,-0.0566,0.0711,Non
concentration_difficulty,0.0457,0.0310,1.4708,0.1420,-0.0153,0.1066,Non
weekly_study_hours,-0.0185,0.0088,-2.0926,0.0369,-0.0358,-0.0011,Oui *


## Données spécifiques aux variables de la première régression :
Si nous nous penchons désormais sur chaque ligne individuelle, on constate des résultats assez intéressants concernant nos 11 variables retenus, car elles n'ont pas toute de l'intérêt. 

$p\text{-value}_j < \alpha \quad (\text{soit } P(\vert{}T\vert{} > \vert{}t_{\text{obs}, j}\vert{}) < 0{,}05) \implies \text{rejet de } H_0^{(j)} \text{ (effet significatif de } X_j \text{)}$

On constate en effet que beaucoup de variables n'ont pas une p-value inférieur au seuil fixé de $\alpha$ = 0.05. Il y a : "average_bedtime_hour", "weekend_sleep_delta_hour", "naps_per_week", "energy_level", "concentration_difficulty", "class_attendance_percent". Ce qui signifie qu'à un niveau de confiance de 95 %, aucune d'entre elles n'a d'effet statistiquement significatif sur le score de sommeil. Au final, cela réduit encore à nouveau le nombre de variables à retenir dans ce modèle, passant de 11 variables explicatives quantitatives (sur 23 variables initiales) à plus que 5 variables significatives (avg_sleep_hours_weekday, pre_sleep_screen_time_min, caffeine_drinks_per_day, weekly_study_hours et cgpa). Notre nouveau modèle de régression linéaireà tester sera donc de la forme :

$$\text{sleep\_score}_i = \beta_0 + \beta_1\,\text{sleep\_wkd}_i + \beta_2\,\text{screen\_min}_i + \beta_3\,\text{caffeine}_i + \beta_4\,\text{study\_h}_i + \beta_5\,\text{cgpa}_i + \varepsilon_i$$

Ainsi, l'objectif sera donc de tester, après la présentation de ce modèle simplifié, les nouvelles valeurs de notre régression.

# Deuxième régression :

In [ ]:
import pandas as pd
import statsmodels.api as sm
vars_reduites = [
    "avg_sleep_hours_weekday",
    "pre_sleep_screen_time_min",
    "caffeine_drinks_per_day",
    "weekly_study_hours",
    "cgpa",
]

X_red = sm.add_constant(df[vars_reduites], has_constant="add")
y = df["sleep_quality_score"]

results_red = sm.OLS(y, X_red).fit()
print(results_red.summary())

ic_95 = results_red.conf_int(alpha=0.05)
df_coef_red = pd.DataFrame({
    "Coefficient": results_red.params,
    "Erreur type": results_red.bse,
    "Statistique t": results_red.tvalues,
    "p-value": results_red.pvalues,
    "IC 95% inf": ic_95[0],
    "IC 95% sup": ic_95[1],
})

df_coef_red["Significatif (5%)"] = df_coef_red["p-value"].apply(
    lambda p: "Oui ***" if p < 0.001 else ("Oui *" if p < 0.05 else "Non")
)

df_affichage_red = df_coef_red.copy()
cols_num = ["Coefficient", "Erreur type", "Statistique t", "IC 95% inf", "IC 95% sup"]
df_affichage_red[cols_num] = df_affichage_red[cols_num].round(4)
df_affichage_red["p-value"] = df_affichage_red["p-value"].apply(
    lambda p: "< 0.0001" if p < 0.0001 else f"{p:.4f}"
)

df_affichage_red

                             OLS Regression Results                            
Dep. Variable:     sleep_quality_score   R-squared:                       0.638
Model:                             OLS   Adj. R-squared:                  0.634
Method:                  Least Squares   F-statistic:                     174.1
Date:                 Tue, 24 Nov 2026   Prob (F-statistic):          1.60e-106
Time:                         02:53:34   Log-Likelihood:                -831.61
No. Observations:                  500   AIC:                             1675.
Df Residuals:                      494   BIC:                             1701.
Df Model:                            5                                         
Covariance Type:             nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const       

,Coefficient,Erreur type,Statistique t,p-value,IC 95% inf,IC 95% sup,Significatif (5%)
const,-0.1047,0.5580,-0.1875,0.8513,-1.2011,0.9918,Non
avg_sleep_hours_weekday,0.8503,0.0469,18.1128,< 0.0001,0.7581,0.9426,Oui ***
pre_sleep_screen_time_min,-0.0200,0.0026,-7.7858,< 0.0001,-0.0251,-0.0150,Oui ***
caffeine_drinks_per_day,-0.1973,0.0460,-4.2868,< 0.0001,-0.2878,-0.1069,Oui ***
weekly_study_hours,-0.0178,0.0088,-2.0274,0.0432,-0.0351,-0.0005,Oui *
cgpa,1.1035,0.2037,5.4178,< 0.0001,0.7033,1.5037,Oui ***


# Analyse des résultats 
## donnée globale la seconde régression :
Tout d'abord concernant le R² ajusté, on retrouve exactement la même valeur de 64.3% de la variance de nos variables explicatives expliquent bien notre variables d'intérêt. En revanche l'AIC, qui était auparavant de 1680 est désormais de 1675. Le changement est très petit mais en comparant deux modèles, on améliore nos résultats quand justement cet AIC diminue (ce qui reste le cas malgré une différence peu importante à première vue).

## donnée spécififique aux variables de la deuxième régression :
Concernant nos vaiables explicatives prises individuellement, nous avons bien réussi à avoir ce que nous voulions, c'est-à-dire, toutes nos variables explicatives tendent donc à avoir une p-value strictement bien inférieur à 0.05 et sont donc bien significtives pour influencer le score de sommeil.